<a href="https://colab.research.google.com/github/liminalvoid/nlp/blob/main/sem_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 7. Fine-tuning BERT

## Установка, импорт библиотек и базовые настройки

In [2]:
%pip -q install -U transformers datasets evaluate accelerate seqeval scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.8 MB/s eta 0:00:00


In [13]:
import os
import random

import numpy as np
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    DataCollatorWithPadding,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)


SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", device)
if torch.cuda.is_available():
    print("GPU model:", torch.cuda.get_device_name(0))

device: cpu


## Sequence classification

В качестве исходного датасета используется набор данных для классификации новостных текстов на русском ([data-silence/rus_news_classifiera](https://huggingface.co/datasets/data-silence/rus_news_classifier)).

In [7]:
dataset_name = "data-silence/rus_news_classifier"
dataset = load_dataset(dataset_name)

print(dataset)

for split in ["train", "test"]:
    print(f"\n--- {split} examples ---")

    for i in range(3):
        example = dataset[split][i]

        print(
            f"[{i}] sentence = {example["news"]}"
            f"    label    = {example["labels"]}"
        )

DatasetDict({
    train: Dataset({
        features: ['news', 'labels'],
        num_rows: 57530
    })
    test: Dataset({
        features: ['news', 'labels'],
        num_rows: 14383
    })
})

--- train examples ---
[0] sentence = Житель Москвы сходил на сеанс эротического массажа, после которого умер. Об этом сообщает Telegram-канал Mash. По информации издания, 31-летний москвич заказывал сеанс массажа с последующими интимными услугами в квартире на бульваре Яна Райниса. Через некоторое время тело мужчины обнаружили в ванной. Жители дома рассказали полиции, что квартира сдавалась посуточно, и в ней регулярно проходили не только аналогичные сеансы, но и вечеринки, сообщает канал. Точную причину смерти москвича определят следователи. Ранее трое жителей Чечни, чьи личности не раскрываются, пропали после оргии в Грозном. Во время вечеринки компания из четырех человек активно снимала видео, в том числе процесс употребления неизвестного порошка.    label    = 1
[1] sentence = В 2021 год

Ниже представлена токенизация и препроцессинг исходного корпуса.

In [10]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoints)
preprocess = lambda batch: tokenizer(
    batch["news"],
    truncation=True,
    max_length=128,
)

tokenized_dataset = dataset.map(preprocess, batched=True)

tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['news', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 57530
    })
    test: Dataset({
        features: ['news', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 14383
    })
})

Загрузка BERT с classification head и вывод модели.

In [11]:
label2id = {
    "climate": 0,
    "conflicts": 1,
    "culture": 2,
    "economy": 3,
    "gloss": 4,
    "health": 5,
    "politics": 6,
    "science": 7,
    "society": 8,
    "sports": 9,
    "travel": 10,
}
id2label = {v: k for k, v in label2id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

print(model)
print("\nClassifier head:")
print(model.classifier)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Метрики для оценки результата.

In [12]:
def calculate_metrics(eval_pred, accuracy_matric, f1_metric):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(
        predictions=preds,
        references=labels,
    )["accuracy"]
    f1 = f1_metric.compute(
        predictions=preds,
        references=labels,
        average="binary",
    )["f1"]

    return {"accuracy": acc, "f1": f1}


accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

Fine-tuning на исходном датасете.

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

selected_range = range(100)
train_dataset = tokenized_dataset["train"].select(selected_range)
test_dataset = tokenized_dataset["test"].select(selected_range)

steps_per_epoch = len(train_dataset) // 16
eval_steps = 5

print("Приблизительное количество шагов за эпоху:", steps_per_epoch)
print("Валидация каждые", eval_steps, "шагов")

trainint_args = TrainingArguments(
    output_dir="output",
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    logging_strategy="steps",
    logging_steps=1,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    loadl_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()